In [1]:
import pandas as pd 
import numpy as np 
import parquet

# maps
import geopandas as gpd
import json
import branca.colormap as cm
import folium
from folium.plugins import TimestampedGeoJson

# visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns 
import matplotlib.pyplot as plt 



In [ ]:
# add daily rides 
# check numbers for 2016 (too litle trips)
# trip duration in 202 249 missings
# in 24-26 trip_duration=0 included

# где какие сатнции наиболее эффективны, а где избыток. stations capacity meet demand
# Где произошла канибализация, установили больше станций но в общей сложности кол-во райдс не увеличилось
# take the oldest stations and look at their behaviour during 10 years. 

# top-5 stations and top-5 routs. 

# ось времени - первые станции в 5 рафонах: можно таблицой внизу дот мэп

# seasonality 
# users behaviour: Casual/Member


# ML
# can we predict demand?
# can we cluster the stations? 



In [80]:
# predict the future. as a time series

In [356]:
from pathlib import Path

DATA_DIR = Path('data')


trips = pd.read_parquet(DATA_DIR / 'trips', columns=['trip_id', 'start_station_id', 'year'])
trips['year'] = trips['year'].astype(int)  

stations = pd.read_parquet(DATA_DIR / 'stations')[['station_id', 'name', 'neighbourhood', 'ward']]

times = pd.read_parquet(DATA_DIR / 'trips', columns=['trip_id', 'year', 'start_time', 'end_time', 'trip_duration'])
times.head()




,trip_id,year,start_time,end_time,trip_duration
0,53279,2016,2016-07-09 01:03:00,2016-07-09 01:15:00,714
1,53394,2016,2016-07-09 02:15:00,2016-07-09 02:22:00,417
2,58314,2016,2016-07-10 17:04:00,2016-07-10 17:36:00,1904
3,60784,2016,2016-07-11 01:45:00,2016-07-11 01:58:00,784
4,93164,2016,2016-07-18 13:35:00,2016-07-18 13:42:00,443


In [364]:
df = trips.merge(
    stations.rename(columns={
        'station_id': 'start_station_id', 'name': 'start_station_name',
        'neighbourhood': 'start_neighbourhood', 'ward': 'start_ward',
    }),
    on='start_station_id', how='left',
)

In [209]:
df.head()

,trip_id,start_station_id,year,start_station_name,start_neighbourhood,start_ward
0,53279,7100,2016,Dundas St E / Regent Park Blvd,Regent Park,Toronto Centre
1,53394,7097,2016,Riverdale Park North (Broadview Ave),North Riverdale,Toronto-Danforth
2,58314,7100,2016,Dundas St E / Regent Park Blvd,Regent Park,Toronto Centre
3,60784,7033,2016,Union Station,St Lawrence-East Bayfront-The Islands,Spadina-Fort York
4,93164,7059,2016,Front St W / Blue Jays Way,Harbourfront-CityPlace,Spadina-Fort York


In [365]:
wards = gpd.read_parquet(DATA_DIR / 'wards' / 'city=toronto' / 'part.parquet')

In [6]:
# print(wards[['ward']].sort_values('ward'))
# print(wards.crs)

### average/median trip duration

In [148]:
times.head()

,trip_id,year,start_time,end_time,trip_duration
0,53279,2016,2016-07-09 01:03:00,2016-07-09 01:15:00,714
1,53394,2016,2016-07-09 02:15:00,2016-07-09 02:22:00,417
2,58314,2016,2016-07-10 17:04:00,2016-07-10 17:36:00,1904
3,60784,2016,2016-07-11 01:45:00,2016-07-11 01:58:00,784
4,93164,2016,2016-07-18 13:35:00,2016-07-18 13:42:00,443


In [376]:
times.isna().groupby(times['year']).sum()

,trip_id,year,start_time,end_time,trip_duration
year,,,,,
2016,0,0,0,0,0
2017,0,0,0,0,0
2018,0,0,0,0,0
2019,0,0,0,0,0
2020,0,0,0,0,0
2021,0,0,0,0,0
2022,0,0,0,0,0
2023,0,0,0,0,0
2024,0,0,0,0,0


In [26]:
times.dtypes

trip_id                   Int64
year                   category
start_time       datetime64[us]
end_time         datetime64[us]
trip_duration             Int32
dtype: object

In [387]:
trip_duration_stats = (times
    .groupby('year', as_index=False)
    .agg(mean_duration = ('trip_duration', 'mean'),
         median_duration = ('trip_duration', 'median'),
         std_duration = ('trip_duration', 'std')
    ))
duration_cols = ['mean_duration', 'median_duration', 'std_duration']
trip_duration_stats[[f'{c}_sec' for c in duration_cols]] = trip_duration_stats[duration_cols].round(0)
trip_duration_stats[[f'{c}_min' for c in duration_cols]] = (trip_duration_stats[duration_cols]/60).round(0)

In [ ]:
trip_duration_stats[['year'] + [f'{c}_min' for c in duration_cols]]


,year,mean_duration_min,median_duration_min,std_duration_min
0,2016,16.0,11.0,20.0
1,2017,17.0,11.0,163.0
2,2018,16.0,11.0,27.0
3,2019,17.0,11.0,309.0
4,2020,20.0,14.0,127.0
5,2021,17.0,13.0,73.0
6,2022,16.0,12.0,75.0
7,2023,18.0,11.0,147.0
8,2024,16.0,11.0,90.0
9,2025,16.0,11.0,24.0


In [ ]:
# mean/median duration is quite steady through the years. 

# Some Vanity metrics (absolute values)

## Stations

In [366]:
stations_yearly = trips.groupby('year')['start_station_id'].nunique().reset_index(name='nr_stations')

In [ ]:
stations_yearly = stations_yearly.assign(
    yoy_change=lambda d: d['nr_stations'].diff(),
    yoy_pct_change=lambda d: d['nr_stations'].pct_change(),
)

stations_count_labels = [f"{v}" for v in stations_yearly['nr_stations']]
stations_pct_labels = [
    '' if pd.isna(p) else f"{p:+.0%}"
    for p in stations_yearly['yoy_pct_change']
]
# below 10% YoY growth reads as a slowdown (navy); 10%+ as healthy growth (orange) -
# warm/cool rather than red/green, which is one of the least colorblind-safe
# pairings
stations_pct_colors = [
    '#888888' if pd.isna(p) else ('#1B3A5C' if p < 0.10 else '#FF6B4A')
    for p in stations_yearly['yoy_pct_change']
]
yoy_hover = [
    '' if pd.isna(v) else f"<br><b>YoY change:</b> {v:+.0f}"
    for v in stations_yearly['yoy_change']
]

fig_stations_per_year = go.Figure()

fig_stations_per_year.add_trace(go.Scatter(
    x=stations_yearly['year'],
    y=stations_yearly['nr_stations'],
    mode='lines+markers+text',
    text=stations_count_labels,
    textposition='top center',
    cliponaxis=False,
    line=dict(color="#FF6B4A", width=3),
    marker=dict(color='#FF6B4A', size=10, line=dict(color='white', width=2)),
    customdata=yoy_hover,
    hovertemplate='<b>Year:</b> %{x}<br><b>Value:</b> %{y:,.0f}%{customdata}<extra></extra>',
))

pct_offset = (stations_yearly['nr_stations'].max() - stations_yearly['nr_stations'].min()) * 0.09

fig_stations_per_year.add_trace(go.Scatter(
    x=stations_yearly['year'],
    y=stations_yearly['nr_stations'] - pct_offset,
    mode='text',
    text=stations_pct_labels,
    textposition='bottom center',
    cliponaxis=False,
    textfont=dict(color=stations_pct_colors, size=11),
    hoverinfo='skip',
    showlegend=False,
))

fig_stations_per_year.update_layout(
    title=dict(text='Stations Per Year', x=0.5),
    xaxis=dict(title='Year', tickmode='array', showgrid=False, showline=True, linecolor='black',tickvals=stations_yearly['year']),
    yaxis=dict(title='Stations', showgrid=False, showline=True, linecolor='black', range=[0, stations_yearly['nr_stations'].max() * 1.18]),
    showlegend=False,
    hoverlabel=dict(bgcolor='#FFF8E7', bordercolor='#D9A440', font=dict(color='#1a1a1a', size=13)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    autosize=True,
    margin=dict(t=100),
)
fig_stations_per_year

## Trips

In [ ]:
# There have been more than 5.3 million trips so far in 2024 (data from January to September)
# That's an average of 19,500 trips per day.
# That's 50% more trips per day than just 2022, and almost double the average in 2021.

In [368]:
trips_yearly = (trips
                    .groupby('year')['start_station_id']
                    .size()
                    .reset_index(name='nr_trips')
                    )

In [ ]:
# trips_yearly.loc[trips_yearly['year']==2016, 'nr_trips'] = 830000

In [559]:
trips_yearly = trips_yearly.assign(yoy_pct_change=lambda d: d['nr_trips'].pct_change())

trips_main = trips_yearly[trips_yearly['year'] <= 2025]
trips_connector = trips_yearly[trips_yearly['year'].isin([2025, 2026])]
trips_last_point = trips_yearly[trips_yearly['year'] == 2026]

trips_pct_offset = (trips_yearly['nr_trips'].max() - trips_yearly['nr_trips'].min()) * 0.09

main_pct_labels = ['' if pd.isna(p) else f"{p:+.0%}" for p in trips_main['yoy_pct_change']]
last_pct_labels = ['' if pd.isna(p) else f"{p:+.0%}" for p in trips_last_point['yoy_pct_change']]

main_yoy_hover = ['' if pd.isna(p) else f"<br><b>YoY change:</b> {p:+.0%}" for p in trips_main['yoy_pct_change']]
last_yoy_hover = ['' if pd.isna(p) else f"<br><b>YoY change:</b> {p:+.0%}" for p in trips_last_point['yoy_pct_change']]

# 2017's YoY % is measured against 2016, a partial (Q3-Q4 only) launch year,
# so it reads as a bogus +155% spike rather than organic growth - blank it,
# same treatment as the cannibalization chart's first point
main_pct_labels[1] = ''
main_yoy_hover[1] = ''

fig_trips_per_year = go.Figure()

fig_trips_per_year.add_trace(go.Scatter(
    x=trips_main['year'],
    y=trips_main['nr_trips'],
    mode='lines+markers+text',
    text=[f"{v/1e6:.1f}M" for v in trips_main['nr_trips']],
    textposition='top center',
    cliponaxis=False,
    line=dict(color="#192562", width=3),
    marker=dict(color='#192562', size=10, line=dict(color='white', width=2)),
    customdata=main_yoy_hover,
    hovertemplate='<b>Year:</b> %{x}<br><b>Value:</b> %{y:,.0f}%{customdata}<extra></extra>',
))

fig_trips_per_year.add_trace(go.Scatter(
    x=trips_connector['year'],
    y=trips_connector['nr_trips'],
    mode='lines',
    line=dict(color='#B0B5BA', width=3, dash='dash'),
    showlegend=False,
    hoverinfo='skip',
))

fig_trips_per_year.add_trace(go.Scatter(
    x=trips_last_point['year'],
    y=trips_last_point['nr_trips'],
    mode='markers+text',
    text=[f"{v/1e6:.1f}M" for v in trips_last_point['nr_trips']],
    textposition='top center',
    cliponaxis=False,
    marker=dict(color='#B0B5BA', size=10, line=dict(color='white', width=2)),
    customdata=last_yoy_hover,
    hovertemplate='<b>Year:</b> %{x}<br><b>Value:</b> %{y:,.0f}%{customdata}<extra></extra>',
    showlegend=False,
))

# YoY % change label under the line, same treatment as the Stations Per Year chart:
# real years in navy, the partial 2026 year in grey to match its muted line/marker
fig_trips_per_year.add_trace(go.Scatter(
    x=trips_main['year'],
    y=trips_main['nr_trips'] - trips_pct_offset,
    mode='text',
    text=main_pct_labels,
    textposition='bottom center',
    cliponaxis=False,
    textfont=dict(color='#1B3A5C', size=11),
    hoverinfo='skip',
    showlegend=False,
))

fig_trips_per_year.add_trace(go.Scatter(
    x=trips_last_point['year'],
    y=trips_last_point['nr_trips'] - trips_pct_offset,
    mode='text',
    text=last_pct_labels,
    textposition='bottom center',
    cliponaxis=False,
    textfont=dict(color='#B0B5BA', size=11),
    hoverinfo='skip',
    showlegend=False,
))

fig_trips_per_year.update_layout(
    title=dict(text='Trips Per Year', x=0.5),
    xaxis=dict(title='Year', tickmode='array', showgrid=False, showline=True, linecolor='black',tickvals=stations_yearly['year']),
    yaxis=dict(title='Trips', showgrid=False, showline=True, linecolor='black',  dtick=1000000, ticks='outside', ticklen=6, tickcolor='black'),
    showlegend=False,
    hoverlabel=dict(bgcolor='#FFF8E7', bordercolor='#D9A440', font=dict(color='#1a1a1a', size=13)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    autosize=True,
    margin=dict(t=100),
)

fig_trips_per_year


### trips per station

In [372]:
trips_per_station = (trips
    .groupby('year', as_index = False)
    .agg(stations = ('start_station_id', 'nunique'),
         trips = ('start_station_id', 'size'))
    .assign(trips_per_station = lambda d: round(d['trips']/d['stations']))
    )

# trips_per_station.loc[trips_per_station['year']==2016, 'trips'] = 830000

# trips_per_station = (trips_per_station
#     .assign(trips_per_station = lambda d: round(d['trips']/d['stations']))
#     )



In [373]:
trips_per_station = (trips_per_station
    .assign(yoy_fleet_change = lambda d: d['stations'].diff(),
            yoy_fleet_pct_change = lambda d: d['stations'].pct_change().round(3),
            yoy_trips_change = lambda d: d['trips'].diff(),
            yoy_pct_trips_change = lambda d: d['trips'].pct_change().round(3),
            yoy_tps_change = lambda d: d['trips_per_station'].diff(),
            yoy_pct_tps_change = lambda d: d['trips_per_station'].pct_change().round(3))
    )
trips_per_station

,year,stations,trips,trips_per_station,yoy_fleet_change,yoy_fleet_pct_change,yoy_trips_change,yoy_pct_trips_change,yoy_tps_change,yoy_pct_tps_change
0,2016,200,585530,2928.0,NaN,NaN,<NA>,<NA>,<NA>,<NA>
1,2017,265,1492366,5632.0,65.0,0.325,906836,1.549,2704.0,0.923
2,2018,359,1922955,5356.0,94.0,0.355,430589,0.289,-276.0,-0.049
3,2019,469,2439448,5201.0,110.0,0.306,516493,0.269,-155.0,-0.029
4,2020,611,2909384,4762.0,142.0,0.303,469936,0.193,-439.0,-0.084
5,2021,627,3569606,5693.0,16.0,0.026,660222,0.227,931.0,0.196
6,2022,682,4615878,6768.0,55.0,0.088,1046272,0.293,1075.0,0.189
7,2023,813,5708745,7022.0,131.0,0.192,1092867,0.237,254.0,0.038
8,2024,879,6942221,7898.0,66.0,0.081,1233476,0.216,876.0,0.125
9,2025,1048,7807041,7449.0,169.0,0.192,864820,0.125,-449.0,-0.057


In [ ]:
# Station rollout came in bursts, creating a repeating expand-and-absorb cycle. 
# Fast growth from 2017 to 2020 spread ridership thin, so trips per station fell. 
# Slower expansion in 2021–2022 let demand catch up. 
# Trips per station peaked at 7910 in 2024 as the network neared capacity. 
# A 19.2% fleet expansion in 2025 then eased the pressure. 2026 is excluded because its data is incomplete.

## Export: standalone HTML panels (chart + description) for the site

In [616]:
description_html = '''
<p>Ridership peaked in 2025 at 7.8 million trips — <span class="highlight">21,400</span> trips per day on average, <span class="highlight">more than 5x the 2017 rate</span>.</p>
'''

chart_div = fig_trips_per_year.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_width='100%',
    default_height='100%',
    config={'responsive': True},
)

panel_html = f'''<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  html, body {{ margin: 0; height: 100%; background: #ffffff; }}
  .panel {{
    display: flex;
    align-items: center;
    gap: 32px;
    background: #ffffff;
    padding: 24px;
    box-sizing: border-box;
    height: 100%;
  }}
  .chart {{ order: 1; flex: 3; height: 100%; min-width: 0; }}
  .text {{
    order: 2;
    flex: 2;
    min-width: 0;
    overflow-wrap: break-word;
    color: #1a1a1a;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    font-size: 20px;
    line-height: 1.5;
  }}
  .text p {{ margin: 0 0 20px 0; }}
  .highlight {{ color: #192562; font-weight: 700; }}
</style>
</head>
<body>
  <div class="panel">
    <div class="text">{description_html}</div>
    <div class="chart">{chart_div}</div>
  </div>
</body>
</html>
'''

with open('site/site_assets/trips_per_year_panel.html', 'w') as f:
    f.write(panel_html)

print("saved site/site_assets/trips_per_year_panel.html")

saved site/site_assets/trips_per_year_panel.html


In [617]:
description_html = '''
<p>The network has grown from 200 stations in <strong>2016</strong> to <span class="highlight">1,098</span> in 2026 — <span class="highlight">more than a 5x expansion</span>  in a decade.</p>
<p>The single biggest jump came in <strong>2025</strong> , when <span class="highlight">169</span> new stations went live in one year alone.</p>
'''

chart_div = fig_stations_per_year.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_width='100%',
    default_height='100%',
    config={'responsive': True},
)

panel_html = f'''<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  html, body {{ margin: 0; height: 100%; background: #ffffff; }}
  .panel {{
    display: flex;
    align-items: center;
    gap: 32px;
    background: #ffffff;
    padding: 24px;
    box-sizing: border-box;
    height: 100%;
  }}
  .text {{
    order: 2;
    flex: 2;
    min-width: 0;
    overflow-wrap: break-word;
    color: #1a1a1a;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    font-size: 20px;
    line-height: 1.5;
  }}
  .text p {{ margin: 0 0 20px 0; }}
  .highlight {{ color: #FF6B4A; font-weight: 700; }}
  .chart {{ order: 1; flex: 3; height: 100%; min-width: 0; }}
</style>
</head>
<body>
  <div class="panel">
    <div class="text">{description_html}</div>
    <div class="chart">{chart_div}</div>
  </div>
</body>
</html>
'''

with open('site/site_assets/stations_per_year_panel.html', 'w') as f:
    f.write(panel_html)

print("saved site/site_assets/stations_per_year_panel.html")

saved site/site_assets/stations_per_year_panel.html


In [619]:
# orange = expansion year (fleet grew 10%+, same threshold as the Stations panel)
# that likely diluted demand; navy = a slower/absorption year where trips per
# station had room to recover. warm/cool rather than red/green, which is one of
# the least colorblind-safe pairings
bar_colors = [
    '#B0B5BA' if y in (2016, 2026) else ('#FF6B4A' if p >= 0.10 else '#1B3A5C')
    for y, p in zip(trips_per_station['year'], trips_per_station['yoy_fleet_pct_change'])
]

cycle_stage = [
    'Incomplete year' if y in (2016, 2026) else (
        'Expansion — fleet growing fast' if p >= 0.10 else 'Absorption — demand catching up'
    )
    for y, p in zip(trips_per_station['year'], trips_per_station['yoy_fleet_pct_change'])
]
bar_opacities = [1.0 if y == 2024 else (0.6 if y in (2016, 2026) else 0.45) for y in trips_per_station['year']]

fig_tps_solo = go.Figure(go.Bar(
    x=trips_per_station['year'],
    y=trips_per_station['trips_per_station'],
    marker=dict(color=bar_colors, opacity=bar_opacities),
    customdata=cycle_stage,
    hovertemplate='<b>Year:</b> %{x}<br><b>Value:</b> %{y:,.0f}<br><b>Stage:</b> %{customdata}<extra></extra>',
))

fig_tps_solo.update_layout(
    title=dict(text='Rides Per Station, by Year', x=0.5),
    xaxis=dict(title='Year', tickmode='array', showgrid=False, showline=True, linecolor='black', tickvals=trips_per_station['year']),
    yaxis=dict(title='Trips per station', showgrid=False, showline=True, linecolor='black'),
    showlegend=False,
    hoverlabel=dict(bgcolor='#FFF8E7', bordercolor='#D9A440', font=dict(color='#1a1a1a', size=13)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    autosize=True,
    bargap=0.5,
)

description_html = '''
<p>The <span class="highlight">expand</span>/<span class="highlight" style="color:#1B3A5C;">absorb</span> cycle shows up directly: years the fleet grows fast (orange) dilute demand per station; slower years (navy) let it catch back up.</p>
<p>Rides per station peaked at <span class="highlight">7,910</span> in 2024, the year fleet growth slowed the most.</p>
<p>Each expansion wave (2017–2020, then 2025) briefly diluted demand across more stations, pulling the number back down before it recovered.</p>
'''

chart_div = fig_tps_solo.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_width='100%',
    default_height='100%',
    config={'responsive': True},
)

panel_html = f'''<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  html, body {{ margin: 0; height: 100%; background: #ffffff; }}
  .panel {{
    display: flex;
    align-items: center;
    gap: 32px;
    background: #ffffff;
    padding: 24px;
    box-sizing: border-box;
    height: 100%;
  }}
  .chart {{ order: 1; flex: 3; height: 100%; min-width: 0; }}
  .text {{
    order: 2;
    flex: 2;
    min-width: 0;
    overflow-wrap: break-word;
    color: #1a1a1a;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    font-size: 20px;
    line-height: 1.5;
  }}
  .text p {{ margin: 0 0 20px 0; }}
  .highlight {{ color: #FF6B4A; font-weight: 700; }}
</style>
</head>
<body>
  <div class="panel">
    <div class="text">{description_html}</div>
    <div class="chart">{chart_div}</div>
  </div>
</body>
</html>
'''

with open('site/site_assets/trips_per_station_panel.html', 'w') as f:
    f.write(panel_html)

print("saved site/site_assets/trips_per_station_panel.html")

saved site/site_assets/trips_per_station_panel.html


In [620]:
# plain data-table fallback for the three Fleet Expansion panels (Trips,
# Stations, Trips per Station), all drawn from the same trips_per_station
# frame - accessibility pass per the dataviz skill: a non-visual view should
# exist alongside the charts
def _pct(p):
    return '' if pd.isna(p) else f"{p:+.0%}"

rows_html = []
for _, r in trips_per_station.iterrows():
    rows_html.append(f'''<tr>
    <td>{int(r['year'])}</td>
    <td>{r['trips']/1e6:.1f}M</td>
    <td>{_pct(r['yoy_pct_trips_change'])}</td>
    <td>{int(r['stations']):,}</td>
    <td>{_pct(r['yoy_fleet_pct_change'])}</td>
    <td>{int(r['trips_per_station']):,}</td>
  </tr>''')

table_html = f'''<details class="growth-table" style="margin: 24px 0;">
  <summary style="cursor:pointer; font-size:16px; color:#555;">View the data behind these charts</summary>
  <style>
    .growth-table table {{ width:100%; border-collapse:collapse; margin-top:16px; font-size:15px; color:#1a1a1a; }}
    .growth-table th {{ text-align:left; color:#555; font-weight:600; padding:6px 12px 6px 0; border-bottom:2px solid #B0B5BA; }}
    .growth-table td {{ padding:6px 12px 6px 0; border-bottom:1px solid #eee; }}
  </style>
  <table>
    <thead>
      <tr>
        <th>Year</th>
        <th>Trips</th>
        <th>YoY % (trips)</th>
        <th>Stations</th>
        <th>YoY % (fleet)</th>
        <th>Trips per station</th>
      </tr>
    </thead>
    <tbody>
      {"".join(rows_html)}
    </tbody>
  </table>
</details>'''

with open('site/site_assets/growth_data_table.html', 'w') as f:
    f.write(table_html)

print("saved site/site_assets/growth_data_table.html")


saved site/site_assets/growth_data_table.html


## MAPS


###  STATIONS MAP (dot timelapse)

In [237]:
stations_full = pd.read_parquet(DATA_DIR / 'stations')[['station_id', 'name', 'lat', 'lon']]
first_year = trips.groupby('start_station_id')['year'].min().rename('first_year')
total_trips = trips.groupby('start_station_id').size().rename('total_trips')


stations_tl = (
    stations_full.set_index('station_id')
    .join(first_year, how='inner')
    .join(total_trips, how='inner')
    .dropna(subset=['lat', 'lon'])
    .reset_index()
)
print(stations_tl.shape, '/', stations_full.shape[0], 'stations plotted')
print(stations_tl['first_year'].value_counts().sort_index())

(1192, 6) / 1202 stations plotted
first_year
2016    200
2017     65
2018     93
2019    107
2020    146
2021     18
2022     66
2023    151
2024     88
2025    177
2026     81
Name: count, dtype: int64


In [339]:
# stations_full.head()
# first year - how many wards
geo_expansion = (df
    .dropna()            
    .groupby('year', as_index=False)
    .agg(unique_neighs=('start_neighbourhood', 'nunique'),
         unique_wards=('start_ward', 'nunique'))
    .assign(change_neighs = lambda d: d['unique_neighs'].diff(),
            change_wards = lambda d: d['unique_wards'].diff())
    )


In [340]:
geo_expansion

,year,unique_neighs,unique_wards,change_neighs,change_wards
0,2016,36,8,NaN,NaN
1,2017,39,10,3.0,2.0
2,2018,49,12,10.0,2.0
3,2019,58,13,9.0,1.0
4,2020,80,19,22.0,6.0
5,2021,80,19,0.0,0.0
6,2022,84,19,4.0,0.0
7,2023,107,21,23.0,2.0
8,2024,124,23,17.0,2.0
9,2025,140,25,16.0,2.0


In [346]:
# print(*df.loc[df['year']==2016, 'start_ward'].unique())
wards_16 = df.loc[df['year']==2016, 'start_ward'].unique()
wards_17 = df.loc[df['year']==2017, 'start_ward'].unique()
print('year_17:', set(wards_17).difference(set(wards_16)))

wards_18 = df.loc[df['year']==2018, 'start_ward'].unique()
print('year_18:', set(wards_18).difference(set(wards_17)))

wards_19 = df.loc[df['year']==2019, 'start_ward'].unique()
print('year_19:', set(wards_19).difference(set(wards_18)))

wards_20 = df.loc[df['year']==2020, 'start_ward'].unique()
print('year_20:', set(wards_20).difference(set(wards_19)))

wards_23 = df.loc[df['year']==2023, 'start_ward'].unique()
print('year_23:', set(wards_23).difference(set(wards_20)))

wards_24 = df.loc[df['year']==2024, 'start_ward'].unique()
print('year_24:', set(wards_24).difference(set(wards_23)))

wards_25 = df.loc[df['year']==2025, 'start_ward'].unique()
print('year_25:', set(wards_25).difference(set(wards_24)))

year_17: {'Scarborough Southwest', 'Etobicoke-Lakeshore'}
year_18: {'Don Valley West', 'Eglinton-Lawrence'}
year_19: {'Don Valley East'}
year_20: {'Scarborough-Guildwood', 'Scarborough-Rouge Park', 'York Centre', 'Humber River-Black Creek', 'Etobicoke Centre', 'York South-Weston'}
year_23: {'Etobicoke North', 'Willowdale'}
year_24: {'Scarborough Centre', 'Don Valley North'}
year_25: {'Scarborough-Agincourt', nan, 'Scarborough North'}


### STATIONS MAP colored by first year - actual version in overview

In [293]:
# on-brand sequential ramp (light -> dark orange), one hue, mapped to first_year
colormap_years = cm.LinearColormap(
    colors=['#FFE0D6', '#FF6B4A', '#8B2E1A'],
    vmin=stations_tl['first_year'].min(),
    vmax=stations_tl['first_year'].max(),
)
colormap_years.caption = 'First year'

station_features_colored = []
for row in stations_tl.itertuples():
    station_features_colored.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [float(row.lon), float(row.lat)]},
        "properties": {
            "times": [f"{row.first_year}-01-01T12:00:00"],
            "icon": "circle",
            "iconstyle": {
                "fillColor": colormap_years(row.first_year), "fillOpacity": 0.8,
                "stroke": True, "color": "white", "weight": 1,
                "radius": float(row.radius_px),
            },
            "tooltip": f"{row.name} ({row.first_year})",
        },
    })

station_features_colored.append({
    "type": "Feature",
    "geometry": {"type": "Point", "coordinates": [float(stations_tl['lon'].iloc[0]), float(stations_tl['lat'].iloc[0])]},
    "properties": {
        "times": [f"{last_year}-12-31T12:00:00"],
        "icon": "circle",
        "iconstyle": {"radius": 0, "fillOpacity": 0, "opacity": 0},
    },
})

geo_stations_timed_colored = {"type": "FeatureCollection", "features": station_features_colored}
print(len(station_features_colored), "colored station features built")

1193 colored station features built


In [ ]:
m_colored = folium.Map(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ',
    zoom_control=False,
    scrollWheelZoom=False,
    doubleClickZoom=False,
    touchZoom=False,
    boxZoom=False,
    keyboard=False,
)
m_colored.fit_bounds(bounds)

TimestampedGeoJson(
    geo_stations_timed_colored,
    period='P1Y',
    duration=None,
    auto_play=True,
    loop=True,
    date_options='YYYY',
    transition_time=600,
).add_to(m_colored)

colormap_years.add_to(m_colored)

m_colored.save('site/site_assets/stations_map_timelapse_by_year.html')


with open('site/site_assets/stations_map_timelapse_by_year.html') as f:
    html = f.read()

html = html.replace('.tickSize(1)', '.tickSize(1)\n        .tickFormat(d3.format("d"))')

with open('site/site_assets/stations_map_timelapse_by_year.html', 'w') as f:
    f.write(html)

print("saved site/site_assets/stations_map_timelapse_by_year.html")

### How the oldest stations behave. Does new stations around "eat" the usage of the oldest ones?

In [400]:
oldest_active_stations = set(df.loc[df['year']==2016, 'start_station_id'].unique()).intersection(set(df.loc[df['year']==2026, 'start_station_id'].unique()))

In [402]:
len(oldest_active_stations)

183

In [404]:
oldest_df = df[df.start_station_id.isin(oldest_active_stations)]

In [406]:
oldest_trips = (oldest_df
    .groupby('year', as_index=False)
    .agg(trips = ('trip_id', 'nunique'),
         stations = ('start_station_id', 'nunique'))
    .assign(trips_per_st = lambda d: round(d['trips']/d['stations']))
    )

In [407]:
oldest_trips

,year,trips,stations,trips_per_st
0,2016,532191,183,2908.0
1,2017,1218546,183,6659.0
2,2018,1290473,183,7052.0
3,2019,1323476,183,7232.0
4,2020,1219537,183,6664.0
5,2021,1459727,183,7977.0
6,2022,1905536,182,10470.0
7,2023,2282787,183,12474.0
8,2024,2655537,181,14671.0
9,2025,2676320,182,14705.0


In [408]:
# looks like the number of the rides rise year on year at the same stations. 
# Although the question whether the station capacity isn't improving (number of bikes rise)

## Are new stations cannibalizing the original ones?

In [ ]:
# "Cannibalization": does opening a new station near an existing one just split
# the same riders between them (like a new retail location drawing sales away from
# an existing location of the same chain), rather than creating net-new demand?
# Test it on the oldest cohort: the 200 stations that existed from day one in 2016.
# If their OWN ridership stalls or falls while the system overall keeps growing,
# that's the signature of cannibalization, not just "the average gets diluted
# because the denominator (station count) grew."
cohort_2016_ids = first_year[first_year == 2016].index
cohort_size = len(cohort_2016_ids)

cohort_trips_yearly = (
    trips[trips['start_station_id'].isin(cohort_2016_ids)]
    .groupby('year').size()
)
total_trips_yearly = trips.groupby('year').size()

cannibalization = pd.DataFrame({
    'cohort_trips': cohort_trips_yearly,
    'total_trips': total_trips_yearly,
}).assign(
    cohort_tps=lambda d: (d['cohort_trips'] / cohort_size).round(1),
    cohort_yoy_pct=lambda d: d['cohort_trips'].pct_change(),
    total_yoy_pct=lambda d: d['total_trips'].pct_change(),
    cohort_share=lambda d: (d['cohort_trips'] / d['total_trips']).round(3),
).reset_index().rename(columns={'index': 'year'})

print(f"2016 cohort: {cohort_size} stations")
print(cannibalization)

cannibalization_plot = (
    cannibalization[~cannibalization['year'].isin([2016, 2026])]
    .reset_index(drop=True)
)


cohort_count_labels = [f"{v/1e6:.1f}M" for v in cannibalization_plot['cohort_trips']]
cohort_pct_labels = ['' if pd.isna(p) else f"{p:+.0%}" for p in cannibalization_plot['cohort_yoy_pct']]
cohort_pct_colors = [
    '#888888' if pd.isna(p) else ('#FF6B4A' if p >= 0.10 else '#1B3A5C')
    for p in cannibalization_plot['cohort_yoy_pct']
]
cohort_yoy_hover = [
    '' if pd.isna(p) else f"<br><b>YoY change:</b> {p:+.0%}"
    for p in cannibalization_plot['cohort_yoy_pct']
]

cohort_pct_labels[0] = ""
cohort_yoy_hover[0] = ''

cohort_pct_offset = (cannibalization_plot['cohort_trips'].max() - cannibalization_plot['cohort_trips'].min()) * 0.12

fig_cannibalization = go.Figure()

fig_cannibalization.add_trace(go.Scatter(
    x=cannibalization_plot['year'],
    y=cannibalization_plot['cohort_trips'],
    mode='lines+markers+text',
    text=cohort_count_labels,
    textposition='top center',
    cliponaxis=False,
    line=dict(color='#FF6B4A', width=3),
    marker=dict(color='#FF6B4A', size=10, line=dict(color='white', width=2)),
    customdata=cohort_yoy_hover,
    hovertemplate='<b>Year:</b> %{x}<br><b>Trips:</b> %{y:,.0f}%{customdata}<extra></extra>',
))

fig_cannibalization.add_trace(go.Scatter(
    x=cannibalization_plot['year'],
    y=cannibalization_plot['cohort_trips'] - cohort_pct_offset,
    mode='text',
    text=cohort_pct_labels,
    textposition='bottom center',
    cliponaxis=False,
    textfont=dict(color=cohort_pct_colors, size=11),
    hoverinfo='skip',
    showlegend=False,
))

fig_cannibalization.update_layout(
    title=dict(text='Trips From the Original 2016 Stations, by Year', x=0.5),
    xaxis=dict(title='Year', tickmode='array', tickvals=cannibalization_plot['year'], showgrid=False, showline=True, linecolor='black'),
    yaxis=dict(title='Trips', showgrid=False, showline=True, linecolor='black', range=[0, cannibalization_plot['cohort_trips'].max() * 1.18]),
    showlegend=False,
    hoverlabel=dict(bgcolor='#FFF8E7', bordercolor='#D9A440', font=dict(color='#1a1a1a', size=13)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    autosize=True,
    margin=dict(t=100),
)

# note the COVID dip directly on the chart instead of in prose - it's an
# obvious external shock, not part of the cannibalization/expansion story
covid_y = cannibalization_plot.loc[cannibalization_plot['year'] == 2020, 'cohort_trips'].values[0]
fig_cannibalization.add_annotation(
    x=2020, y=covid_y,
    text='COVID disruption',
    showarrow=True, arrowhead=0, ax=0, ay=-40,
    font=dict(color='#888888', size=11),
)

fig_cannibalization

### cannibalization_panel.html

In [636]:
description_html = '''
<p>There’s no clear cannibalization signature. The cohort’s trips grew almost every year, just more slowly than the whole system. That’s expected for a fixed set of 200 stations inside a network that grew about 4x larger.</p>
<p>The only real stall was <span class="highlight">2025 (−0.8%)</span>, the year with the most expansion. That fits the same expand-then-absorb cycle shown in the Trips per Station chart.</p>
'''

chart_div = fig_cannibalization.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_width='100%',
    default_height='100%',
    config={'responsive': True},
)

panel_html = f'''<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  html, body {{ margin: 0; height: 100%; background: #ffffff; }}
  .panel {{
    display: flex;
    align-items: center;
    gap: 32px;
    background: #ffffff;
    padding: 24px;
    box-sizing: border-box;
    height: 100%;
  }}
  .chart {{ order: 1; flex: 3; height: 100%; min-width: 0; }}
  .text {{
    order: 2;
    flex: 2;
    min-width: 0;
    overflow-wrap: break-word;
    color: #1a1a1a;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    font-size: 20px;
    line-height: 1.5;
  }}
  .text p {{ margin: 0 0 20px 0; }}
  .highlight {{ color: #1B3A5C; font-weight: 700; }}
</style>
</head>
<body>
  <div class="panel">
    <div class="text">{description_html}</div>
    <div class="chart">{chart_div}</div>
  </div>
</body>
</html>
'''

with open('site/site_assets/cannibalization_panel.html', 'w') as f:
    f.write(panel_html)

print("saved site/site_assets/cannibalization_panel.html")


saved site/site_assets/cannibalization_panel.html


## Users behaviour and seasonal patterns

In [ ]:
times = pd.read_parquet(DATA_DIR / 'trips', columns=['trip_id', 'year', 'start_time'])
times.head()

In [411]:
times = (times
    .assign(year_month = lambda d: d['start_time'].dt.to_period('M'),
            hour = lambda d: d.start_time.dt.hour)
    )

In [601]:
hourly = (times[times['year']==2026]
    .groupby('hour')
    .size()
    .reset_index(name='trips')
    )
hourly.head()

,hour,trips
0,0,100269
1,1,60504
2,2,45859
3,3,24649
4,4,18158


### hourly plot

In [602]:
hourly_labels = [f"{v/1e3:.0f}K" for v in hourly['trips']]

# rush hour = 8 AM and 4-6 PM; everything else fades to let them pop (same
# treatment as fig_departure)
rush_hours = {8, 16, 17, 18}
bar_opacities = [1.0 if h in rush_hours else 0.3 for h in hourly['hour']]

fig_hourly = go.Figure(go.Bar(
    x=hourly['hour'],
    y=hourly['trips'],
    text=hourly_labels,
    textposition='outside',
    cliponaxis=False,
    marker=dict(color="#FF6B4A", opacity=bar_opacities),
    hovertemplate='<b>Hour:</b> %{x}:00<br><b>Trips:</b> %{y:,.0f}<extra></extra>',
))

fig_hourly.update_layout(
    title=dict(text='Trips by Hour of Day, 2026', x=0.5),
    xaxis=dict(title='Hour of Trip Start', showgrid=False, dtick=1, showline=True, linecolor='black'),
    yaxis=dict(title='Trips', showgrid=False, tickformat='.2s', showline=True, linecolor='black'),
    showlegend=False,
    hoverlabel=dict(bgcolor='#FFF8E7', bordercolor='#D9A440', font=dict(color='#1a1a1a', size=13)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=1100,
    height=500,
)

fig_hourly

### hourly_pattern_panel.html

In [621]:
description_html = '''
<p>At 5 p.m. on almost any weekday, this is what Toronto's bikeshare network looks like: trips surge to <span class="highlight">a sharp evening peak</span>, repeating itself every single day.</p>
<p>Bars shown are 2026 only; hours 8 AM and 4-6 PM (rush hour) are highlighted at full opacity, everything else fades.</p>
'''

chart_div = fig_hourly.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_width='100%',
    default_height='100%',
    config={'responsive': True},
)

panel_html = f'''<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  html, body {{ margin: 0; height: 100%; background: #ffffff; }}
  .panel {{
    display: flex;
    align-items: center;
    gap: 32px;
    background: #ffffff;
    padding: 24px;
    box-sizing: border-box;
    height: 100%;
  }}
  .chart {{ order: 1; flex: 3; height: 100%; min-width: 0; }}
  .text {{
    order: 2;
    flex: 2;
    min-width: 0;
    overflow-wrap: break-word;
    color: #1a1a1a;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    font-size: 20px;
    line-height: 1.5;
  }}
  .text p {{ margin: 0 0 20px 0; }}
  .highlight {{ color: #FF6B4A; font-weight: 700; }}
</style>
</head>
<body>
  <div class="panel">
    <div class="text">{description_html}</div>
    <div class="chart">{chart_div}</div>
  </div>
</body>
</html>
'''

with open('site/site_assets/hourly_pattern_panel.html', 'w') as f:
    f.write(panel_html)

print("saved site/site_assets/hourly_pattern_panel.html")

saved site/site_assets/hourly_pattern_panel.html


In [603]:
hourly_user_type_times = pd.read_parquet(DATA_DIR / 'trips', columns=['year', 'start_time', 'user_type'])
hourly_user_type_times['year'] = hourly_user_type_times['year'].astype(int)

# 2018-2023 label the same two rider classes "Annual Member"/"Casual Member" instead
# of "Member"/"Casual" (see the departure-time section above) - 2026 alone doesn't need
# this, but keeping the mapping makes the cell correct regardless of execution order
USER_TYPE_MAP = {'Annual Member': 'Member', 'Casual Member': 'Casual'}
hourly_user_type_times['user_type'] = hourly_user_type_times['user_type'].astype(str).replace(USER_TYPE_MAP)

hourly_by_type_src = hourly_user_type_times[hourly_user_type_times['year'] == 2026].copy()
hourly_by_type_src['hour'] = hourly_by_type_src['start_time'].dt.hour

hourly_type_counts = (hourly_by_type_src
    .groupby(['user_type', 'hour'])
    .size()
    .reset_index(name='trips')
    )

def hourly_type_series(name):
    return (hourly_type_counts[hourly_type_counts['user_type'] == name]
        .set_index('hour')['trips']
        .reindex(range(24), fill_value=0))

member_hourly = hourly_type_series('Member')
casual_hourly = hourly_type_series('Casual')

In [610]:
recent_years = [2024, 2025, 2026]
departure_src = times[times['year'].astype(int).isin(recent_years)]

# 2-hour bins offset by 1h 
# hour in the middle of its own bin instead of split across two bins
bin_idx = ((departure_src['start_time'].dt.hour + 1) % 24) // 2
bin_start_hour = {i: (23 if i == 0 else 2 * i - 1) for i in range(12)}

def fmt_time(h, minute):
    period = 'AM' if h < 12 else 'PM'
    h12 = h % 12 or 12
    return f"{h12}:{minute} {period}"

bin_labels = {
    i: f"{fmt_time(bin_start_hour[i], '00')} - {fmt_time((bin_start_hour[i] + 1) % 24, '59')}"
    for i in range(12)
}
bin_order = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 0]

departure = (bin_idx
    .groupby(bin_idx)
    .size()
    .reindex(bin_order)
    .reset_index(name='trips')
    .rename(columns={'start_time': 'bin'})
    )
departure['label'] = departure['bin'].map(bin_labels)
departure

,bin,trips,label
0,1,330910,1:00 AM - 2:59 AM
1,2,131137,3:00 AM - 4:59 AM
2,3,429852,5:00 AM - 6:59 AM
3,4,2085648,7:00 AM - 8:59 AM
4,5,1806942,9:00 AM - 10:59 AM
5,6,2044697,11:00 AM - 12:59 PM
6,7,2361073,1:00 PM - 2:59 PM
7,8,3166881,3:00 PM - 4:59 PM
8,9,4224248,5:00 PM - 6:59 PM
9,10,2610504,7:00 PM - 8:59 PM


In [613]:
user_type_times = pd.read_parquet(DATA_DIR / 'trips', columns=['year', 'start_time', 'user_type'])
user_type_times['year'] = user_type_times['year'].astype(int)

# 2018-2023 label the same two rider classes "Annual Member"/"Casual Member" instead
# of "Member"/"Casual" - without this, filtering to 2023-2025 silently drops all of
# 2023 (5.7M trips) since none of its rows match either of the later years' labels
USER_TYPE_MAP = {'Annual Member': 'Member', 'Casual Member': 'Casual'}
user_type_times['user_type'] = user_type_times['user_type'].astype(str).replace(USER_TYPE_MAP)

departure_by_type = user_type_times[user_type_times['year'].isin(recent_years)].copy()
departure_by_type['bin'] = ((departure_by_type['start_time'].dt.hour + 1) % 24) // 2

departure_type_counts = (departure_by_type
    .groupby(['user_type', 'bin'])
    .size()
    .reset_index(name='trips')
    )

def type_series(name):
    return (departure_type_counts[departure_type_counts['user_type'] == name]
        .set_index('bin')['trips']
        .reindex(bin_order, fill_value=0))

member_counts = type_series('Member')
casual_counts = type_series('Casual')
bin_labels_ordered = [bin_labels[b] for b in bin_order]

In [615]:
fig_combined = make_subplots(
    rows=2, cols=2,
    specs=[[{'rowspan': 2}, {}], [None, {}]],
    column_widths=[0.55, 0.45],
    horizontal_spacing=0.08,
    vertical_spacing=0.12,
    subplot_titles=('All Riders', 'Member', 'Casual'),
)

rush_bins = {4, 9}

# left: all riders (same as fig_departure)
bar_opacities_all = [1.0 if b in rush_bins else 0.3 for b in departure['bin']]
rush_hover = ['<br><b>Rush hour</b>' if b in rush_bins else '' for b in departure['bin']]

fig_combined.add_trace(go.Bar(
    x=departure['label'],
    y=departure['trips'],
    marker=dict(color="#FF6B4A", opacity=bar_opacities_all),
    customdata=rush_hover,
    hovertemplate='<b>Time:</b> %{x}<br><b>Trips:</b> %{y:,.0f}%{customdata}<extra></extra>',
), row=1, col=1)

# right: Member / Casual (same as fig_by_type)
bar_opacities_type = [1.0 if b in rush_bins else 0.3 for b in bin_order]

fig_combined.add_trace(go.Bar(
    x=bin_labels_ordered,
    y=member_counts.values,
    marker=dict(color='#D9A440', opacity=bar_opacities_type),
    hovertemplate='<b>Time:</b> %{x}<br><b>Trips:</b> %{y:,.0f}<extra></extra>',
), row=1, col=2)

# casual riders don't have a meaningful AM rush - just the 5-7 PM leisure
# window - so only that bin gets highlighted here, unlike Member's two peaks
bar_opacities_casual = [1.0 if b == 9 else 0.3 for b in bin_order]

fig_combined.add_trace(go.Bar(
    x=bin_labels_ordered,
    y=casual_counts.values,
    marker=dict(color="#1B3A5C", opacity=bar_opacities_casual),
    hovertemplate='<b>Time:</b> %{x}<br><b>Trips:</b> %{y:,.0f}<extra></extra>',
), row=2, col=2)

fig_combined.update_layout(
    title=dict(text='Trips by Departure Time, 2024-2026', x=0.5),
    showlegend=False,
    hoverlabel=dict(bgcolor='#FFF8E7', bordercolor='#D9A440', font=dict(color='#1a1a1a', size=13)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    autosize=True,
    margin=dict(t=100),
)

fig_combined.update_xaxes(title='Time of Trip Start', showgrid=False, showline=True, linecolor='black', tickangle=-45, row=1, col=1)
fig_combined.update_xaxes(showgrid=False, showline=True, linecolor='black', showticklabels=False, ticks='', row=1, col=2)
fig_combined.update_xaxes(title='Time of Trip Start', showgrid=False, showline=True, linecolor='black', tickangle=-45, row=2, col=2)

fig_combined.update_yaxes(title='', showgrid=False, tickformat='.2s', showline=True, linecolor='black', row=1, col=1)
fig_combined.update_yaxes(title='', showgrid=False, tickformat='.2s', showline=True, linecolor='black', row=1, col=2)
fig_combined.update_yaxes(title='', showgrid=False, tickformat='.2s', showline=True, linecolor='black', row=2, col=2)

fig_combined

In [623]:
description_html = ''''''

chart_div = fig_combined.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_width='100%',
    default_height='100%',
    config={'responsive': True},
)

panel_html = f'''<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  html, body {{ margin: 0; height: 100%; background: #ffffff; }}
  .panel {{
    display: flex;
    flex-direction: column;
    gap: 16px;
    background: #ffffff;
    padding: 24px;
    box-sizing: border-box;
    height: 100%;
  }}
  .text {{
    flex: 0 0 auto;
    color: #1a1a1a;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    font-size: 20px;
    line-height: 1.5;
  }}
  .text p {{ margin: 0 0 12px 0; }}
  .highlight {{ color: #FF6B4A; font-weight: 700; }}
  .chart {{ flex: 1 1 auto; min-height: 0; }}
</style>
</head>
<body>
  <div class="panel">
    <div class="text">{description_html}</div>
    <div class="chart">{chart_div}</div>
  </div>
</body>
</html>
'''

with open('site/site_assets/departure_time_panel.html', 'w') as f:
    f.write(panel_html)

print("saved site/site_assets/departure_time_panel.html")

saved site/site_assets/departure_time_panel.html


In [626]:
weekday_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

weekly_type_src = user_type_times[user_type_times['year'].isin(recent_years)].copy()
weekly_type_src['dow'] = weekly_type_src['start_time'].dt.dayofweek

weekly_type_counts = (weekly_type_src
    .groupby(['user_type', 'dow'])
    .size()
    .reset_index(name='trips')
    )

def weekly_type_series(name):
    return (weekly_type_counts[weekly_type_counts['user_type'] == name]
        .set_index('dow')['trips']
        .reindex(range(7), fill_value=0))

member_weekly = weekly_type_series('Member')
casual_weekly = weekly_type_series('Casual')


/var/folders/xf/ygl5_zk57kn1rh2h1xj9fyx40000gn/T/ipykernel_1611/955633181.py:2: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  weekly_type_src['dow'] = weekly_type_src['start_time'].dt.dayofweek


In [628]:
weekend_days = {5, 6}
all_riders_weekly = member_weekly + casual_weekly

fig_weekly_combined = make_subplots(
    rows=2, cols=2,
    specs=[[{'rowspan': 2}, {}], [None, {}]],
    column_widths=[0.55, 0.45],
    horizontal_spacing=0.08,
    vertical_spacing=0.12,
    subplot_titles=('All Riders', 'Member', 'Casual'),
)

# Member commutes on weekdays, so invert the highlight here - weekdays bright,
# weekend faded - opposite of Casual's weekend-bright treatment below. All
# Riders stays full opacity throughout - the combined total doesn't have a
# single clean day-of-week signature of its own (Member/Casual partly cancel
# out), so dimming any of its bars would imply a pattern that isn't really there
bar_opacities_weekend = [1.0 if d in weekend_days else 0.3 for d in range(7)]
bar_opacities_member = [1.0 if d not in weekend_days else 0.3 for d in range(7)]

fig_weekly_combined.add_trace(go.Bar(
    x=weekday_labels,
    y=all_riders_weekly.values,
    marker=dict(color="#FF6B4A", opacity=0.65),
    hovertemplate='<b>Day:</b> %{x}<br><b>Trips:</b> %{y:,.0f}<extra></extra>',
), row=1, col=1)

fig_weekly_combined.add_trace(go.Bar(
    x=weekday_labels,
    y=member_weekly.values,
    marker=dict(color='#D9A440', opacity=bar_opacities_member),
    hovertemplate='<b>Day:</b> %{x}<br><b>Trips:</b> %{y:,.0f}<extra></extra>',
), row=1, col=2)

fig_weekly_combined.add_trace(go.Bar(
    x=weekday_labels,
    y=casual_weekly.values,
    marker=dict(color="#1B3A5C", opacity=bar_opacities_weekend),
    hovertemplate='<b>Day:</b> %{x}<br><b>Trips:</b> %{y:,.0f}<extra></extra>',
), row=2, col=2)

fig_weekly_combined.update_layout(
    title=dict(text='Trips by Day of Week, 2024-2026', x=0.5),
    showlegend=False,
    hoverlabel=dict(bgcolor='#FFF8E7', bordercolor='#D9A440', font=dict(color='#1a1a1a', size=13)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    autosize=True,
    margin=dict(t=100),
)

fig_weekly_combined.update_xaxes(title='Day of Week', showgrid=False, showline=True, linecolor='black', row=1, col=1)
fig_weekly_combined.update_xaxes(showgrid=False, showline=True, linecolor='black', showticklabels=False, ticks='', row=1, col=2)
fig_weekly_combined.update_xaxes(title='Day of Week', showgrid=False, showline=True, linecolor='black', row=2, col=2)

# same y-axis range on Member and Casual so their relative heights are
# directly comparable - this is what makes the weekend crossover (Casual
# bars taller than Member's) visible in the chart itself, not just in the text
type_y_max = max(member_weekly.max(), casual_weekly.max()) * 1.1

fig_weekly_combined.update_yaxes(title='', showgrid=False, tickformat='.2s', showline=True, linecolor='black', row=1, col=1)
fig_weekly_combined.update_yaxes(title='', showgrid=False, tickformat='.2s', showline=True, linecolor='black', range=[0, type_y_max], row=1, col=2)
fig_weekly_combined.update_yaxes(title='', showgrid=False, tickformat='.2s', showline=True, linecolor='black', range=[0, type_y_max], row=2, col=2)

fig_weekly_combined


In [629]:
description_html = ''''''

chart_div = fig_weekly_combined.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_width='100%',
    default_height='100%',
    config={'responsive': True},
)

panel_html = f'''<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  html, body {{ margin: 0; height: 100%; background: #ffffff; }}
  .panel {{
    display: flex;
    flex-direction: column;
    gap: 16px;
    background: #ffffff;
    padding: 24px;
    box-sizing: border-box;
    height: 100%;
  }}
  .text {{
    flex: 0 0 auto;
    color: #1a1a1a;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    font-size: 20px;
    line-height: 1.5;
  }}
  .text p {{ margin: 0 0 12px 0; }}
  .highlight {{ color: #FF6B4A; font-weight: 700; }}
  .chart {{ flex: 1 1 auto; min-height: 0; }}
</style>
</head>
<body>
  <div class="panel">
    <div class="text">{description_html}</div>
    <div class="chart">{chart_div}</div>
  </div>
</body>
</html>
'''

with open('site/site_assets/weekly_type_panel.html', 'w') as f:
    f.write(panel_html)

print("saved site/site_assets/weekly_type_panel.html")


saved site/site_assets/weekly_type_panel.html


In [594]:
# restricted to 2024-2026: 2018-2023's member/casual labeling is unreliable
# (see the monthly-by-type cells below) and the earlier years would force this
# and every seasonal chart derived from `monthly` to mix in that bad window -
# 2024-2026 is the one stretch with both clean labels and full years
monthly = (times[times['year'].astype(int) >= 2024]
    .groupby(['year_month'])['trip_id']
    .size()
    .reset_index(name='nr_trips')
    )
monthly['year'] = monthly['year_month'].dt.year
monthly['rolling_12mo'] = monthly['nr_trips'].rolling(12, min_periods=1).mean()

In [595]:
# 2026 (year-to-date) is an incomplete period, not a real dip - gray it out
# rather than let it read as part of the trend
bar_colors = ['#B0B5BA' if y == 2026 else '#FF6B4A' for y in monthly['year']]

fig_monthly = go.Figure()

fig_monthly.add_trace(go.Bar(
    x=monthly['year_month'].dt.to_timestamp(),
    y=monthly['nr_trips'],
    marker=dict(color=bar_colors),
    opacity=0.9,
    name='Trips',
    hovertemplate='<b>Month:</b> %{x|%b %Y}<br><b>Trips:</b> %{y:,.0f}<extra></extra>',
))

fig_monthly.add_trace(go.Scatter(
    x=monthly['year_month'].dt.to_timestamp(),
    y=monthly['rolling_12mo'],
    mode='lines',
    line=dict(color='#1B3A5C', width=2.5),
    name='12-mo rolling avg',
    hovertemplate='<b>Month:</b> %{x|%b %Y}<br><b>12-mo avg:</b> %{y:,.0f}<extra></extra>',
))

fig_monthly.update_layout(
    title=dict(text='Trips by Month, 2024-2026', x=0.5),
    xaxis=dict(title='Month', showgrid=False, dtick='M12', tickformat='%Y'),
    yaxis=dict(title='Trips', showgrid=False, tickformat='.2s'),
    showlegend=True,
    legend=dict(orientation='h', yanchor='top', y=-0.18, xanchor='center', x=0.5),
    hoverlabel=dict(bgcolor='#FFF8E7', bordercolor='#D9A440', font=dict(color='#1a1a1a', size=13)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    autosize=True,
    height=500,
)

fig_monthly

In [596]:
fig_monthly.write_html(
    'site/site_assets/monthly_trips_panel.html',
    include_plotlyjs='cdn',
    full_html=True,
    default_width='100%',
    default_height='100%',
    config={'responsive': True},
)

print("saved site/site_assets/monthly_trips_panel.html")

saved site/site_assets/monthly_trips_panel.html


In [ ]:
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

seasonality = monthly.copy()
seasonality['calendar_month'] = seasonality['year_month'].dt.month
seasonality


,year_month,nr_trips,year,rolling_12mo,calendar_month
0,2024-01,204696,2024,204696.000000,1
1,2024-02,261542,2024,233119.000000,2
2,2024-03,312303,2024,259513.666667,3
3,2024-04,403352,2024,295473.250000,4
4,2024-05,685651,2024,373508.800000,5
5,2024-06,766118,2024,438943.666667,6
6,2024-07,903573,2024,505319.285714,7
7,2024-08,905173,2024,555301.000000,8
8,2024-09,899886,2024,593588.222222,9
9,2024-10,792841,2024,613513.500000,10


In [599]:
# 2026 is a partial year (see fig_monthly) - dashed gray rather than a color,
# so a partial line never gets mistaken for a real seasonal shape
complete_years = sorted(y for y in seasonality['year'].unique() if y != 2026)
n = len(complete_years)
shades = pc.sample_colorscale('Blues', [0.35 + 0.65 * i / (n - 1) for i in range(n)])
year_colors = dict(zip(complete_years, shades))
year_colors[2026] = '#B0B5BA' 

In [600]:
def fmt_trips(v):
    return f"{v / 1e6:.1f}M" if v >= 1_000_000 else f"{v / 1e3:.0f}k"

fig_seasonality = go.Figure()

# descending by year, not by value - unified hover keeps one fixed row order for every
# hovered month, and trips have grown roughly monotonically year over year, so this
# puts the highest values on top for most months
for year in sorted(seasonality['year'].unique(), reverse=True):
    yearly = seasonality[seasonality['year'] == year].sort_values('calendar_month')
    incomplete = year == 2026
    label = f"{year}{' (partial)' if incomplete else ''}"
    fig_seasonality.add_trace(go.Scatter(
        x=yearly['calendar_month'].map(lambda m: MONTH_NAMES[m - 1]),
        y=yearly['nr_trips'],
        text=[fmt_trips(v) for v in yearly['nr_trips']],
        mode='lines',
        name=label,
        line=dict(color=year_colors[year], width=2, dash='dash' if incomplete else 'solid'),
        hovertemplate=f'{label} - %{{text}}<extra></extra>',
    ))

fig_seasonality.update_layout(
    title=dict(text='Seasonality: Trips by Month, 2024-2026', x=0.5),
    xaxis=dict(
        title='Month', showgrid=False, showline=True, linecolor='black',
        categoryorder='array', categoryarray=MONTH_NAMES,
        showspikes=True, spikemode='across', spikesnap='cursor',
        spikedash='dot', spikethickness=1, spikecolor='#888888',
    ),
    yaxis=dict(title='Trips', showgrid=False, tickformat='.2s', showline=True, linecolor='black'),
    legend=dict(title='Year'),
    hovermode='x unified',
    hoverlabel=dict(bgcolor='#FFF8E7', bordercolor='#D9A440', font=dict(color='#1a1a1a', size=13)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    autosize=True,
    height=600,
)

fig_seasonality

In [630]:
description_html = '''
<p>The shape barely changes year to year, even as 2024-2026 saw steady growth. Every year traces almost the same curve — a spring climb, a summer plateau, a winter drop-off — just larger each time.</p>
<p>2026 (still in progress) is <span class="highlight">grayed out</span> because it's incomplete, not because ridership actually fell.</p>
'''

seasonality_chart_div = fig_seasonality.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_width='100%',
    default_height='100%',
    config={'responsive': True},
)

panel_html = f'''<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  html, body {{ margin: 0; height: 100%; background: #ffffff; }}
  .panel {{
    display: flex;
    flex-direction: column;
    gap: 16px;
    background: #ffffff;
    padding: 24px;
    box-sizing: border-box;
    height: 100%;
  }}
  .text {{
    flex: 0 0 auto;
    color: #1a1a1a;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    font-size: 20px;
    line-height: 1.5;
  }}
  .text p {{ margin: 0 0 12px 0; }}
  .highlight {{ color: #1B3A5C; font-weight: 700; }}
  .chart {{ flex: 1 1 auto; width: 100%; min-height: 0; }}
</style>
</head>
<body>
  <div class="panel">
    <div class="text">{description_html}</div>
    <div class="chart">{seasonality_chart_div}</div>
  </div>
</body>
</html>
'''

with open('site/site_assets/seasonality_panel.html', 'w') as f:
    f.write(panel_html)

print("saved site/site_assets/seasonality_panel.html")

saved site/site_assets/seasonality_panel.html


In [631]:
# monthly trips split by rider type, 2024-2026 (same window as `monthly` -
# member/casual labeling is unreliable before 2024, see the weekly section)
monthly_type_src = user_type_times[user_type_times['year'].astype(int) >= 2024].copy()
monthly_type_src['year_month'] = monthly_type_src['start_time'].dt.to_period('M')

monthly_type_counts = (monthly_type_src
    .groupby(['user_type', 'year_month'])
    .size()
    .reset_index(name='trips')
    )

def monthly_type_series(name):
    return (monthly_type_counts[monthly_type_counts['user_type'] == name]
        .set_index('year_month')['trips']
        .reindex(monthly['year_month'], fill_value=0)
        )

member_monthly = monthly_type_series('Member')
casual_monthly = monthly_type_series('Casual')


In [641]:
month_x = monthly['year_month'].dt.to_timestamp()

# defined locally rather than reused from fig_monthly's cell - 'bar_colors' is a
# generic name reused across many cells in this notebook with different
# lengths/meanings, so relying on cross-cell reuse breaks depending on run order
bar_colors = ['#B0B5BA' if y == 2026 else '#FF6B4A' for y in monthly['year']]

fig_monthly_combined = make_subplots(
    rows=2, cols=2,
    specs=[[{'colspan': 2}, None], [{}, {}]],
    row_heights=[0.5, 0.5],
    horizontal_spacing=0.08,
    vertical_spacing=0.18,
    subplot_titles=('All Riders', 'Member', 'Casual'),
)

# top row: same bars + 12-mo rolling avg as the standalone Trips by Month chart
fig_monthly_combined.add_trace(go.Bar(
    x=month_x,
    y=monthly['nr_trips'],
    marker=dict(color=bar_colors),
    opacity=0.9,
    name='Trips',
    hovertemplate='<b>Month:</b> %{x|%b %Y}<br><b>Trips:</b> %{y:,.0f}<extra></extra>',
), row=1, col=1)

fig_monthly_combined.add_trace(go.Scatter(
    x=month_x,
    y=monthly['rolling_12mo'],
    mode='lines',
    line=dict(color='#1B3A5C', width=2.5),
    name='12-mo rolling avg',
    hovertemplate='<b>Month:</b> %{x|%b %Y}<br><b>12-mo avg:</b> %{y:,.0f}<extra></extra>',
), row=1, col=1)

# bottom row: Member (left), Casual (right)
fig_monthly_combined.add_trace(go.Bar(
    x=month_x,
    y=member_monthly.values,
    marker=dict(color='#D9A440'),
    showlegend=False,
    hovertemplate='<b>Month:</b> %{x|%b %Y}<br><b>Trips:</b> %{y:,.0f}<extra></extra>',
), row=2, col=1)

fig_monthly_combined.add_trace(go.Bar(
    x=month_x,
    y=casual_monthly.values,
    marker=dict(color="#1B3A5C"),
    showlegend=False,
    hovertemplate='<b>Month:</b> %{x|%b %Y}<br><b>Trips:</b> %{y:,.0f}<extra></extra>',
), row=2, col=2)

fig_monthly_combined.update_layout(
    title=dict(text='Trips by Month, 2024-2026', x=0.5),
    showlegend=True,
    legend=dict(orientation='h', yanchor='top', y=-0.1, xanchor='center', x=0.5),
    hoverlabel=dict(bgcolor='#FFF8E7', bordercolor='#D9A440', font=dict(color='#1a1a1a', size=13)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    autosize=True,
    margin=dict(t=100, b=80),
)

fig_monthly_combined.update_xaxes(title='Month', showgrid=False, showline=True, linecolor='black', dtick='M6', tickformat='%b %Y', row=1, col=1)
fig_monthly_combined.update_xaxes(title='Month', showgrid=False, showline=True, linecolor='black', dtick='M6', tickformat='%b %Y', row=2, col=1)
fig_monthly_combined.update_xaxes(title='Month', showgrid=False, showline=True, linecolor='black', dtick='M6', tickformat='%b %Y', row=2, col=2)

# same y-axis range on Member and Casual so their relative heights are directly
# comparable, same reasoning as the weekly combined chart above
type_y_max = max(member_monthly.max(), casual_monthly.max()) * 1.1

fig_monthly_combined.update_yaxes(title='', showgrid=False, tickformat='.2s', showline=True, linecolor='black', row=1, col=1)
fig_monthly_combined.update_yaxes(title='', showgrid=False, tickformat='.2s', showline=True, linecolor='black', range=[0, type_y_max], row=2, col=1)
fig_monthly_combined.update_yaxes(title='', showgrid=False, tickformat='.2s', showline=True, linecolor='black', range=[0, type_y_max], row=2, col=2)

fig_monthly_combined


In [639]:
description_html = '''
<p>Member ridership stays fairly steady year-round — about an <span class="highlight">8x</span> gap between its quietest month and its busiest. Casual ridership swings far more: a <span class="highlight">50x</span> gap between winter and summer.</p>
<p>That gap has been widening too: casual riders made up just 7-9% of winter trips in 2024, but by 2026's summer peak they're over a third of all trips — growing every year.</p>
'''

chart_div = fig_monthly_combined.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_width='100%',
    default_height='100%',
    config={'responsive': True},
)

panel_html = f'''<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  html, body {{ margin: 0; height: 100%; background: #ffffff; }}
  .panel {{
    display: flex;
    flex-direction: column;
    gap: 16px;
    background: #ffffff;
    padding: 24px;
    box-sizing: border-box;
    height: 100%;
  }}
  .text {{
    flex: 0 0 auto;
    color: #1a1a1a;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    font-size: 20px;
    line-height: 1.5;
  }}
  .text p {{ margin: 0 0 12px 0; }}
  .highlight {{ color: #FF6B4A; font-weight: 700; }}
  .chart {{ flex: 1 1 auto; min-height: 0; }}
</style>
</head>
<body>
  <div class="panel">
    <div class="text">{description_html}</div>
    <div class="chart">{chart_div}</div>
  </div>
</body>
</html>
'''

with open('site/site_assets/monthly_type_panel.html', 'w') as f:
    f.write(panel_html)

print("saved site/site_assets/monthly_type_panel.html")


saved site/site_assets/monthly_type_panel.html


In [642]:
# daily version of the monthly combined chart. All Riders isn't affected by the
# member/casual labeling bug (2018-2023, see the weekly/monthly sections above),
# so it can safely use the full history since 2017 (2016 excluded as a partial
# pilot year) - Member/Casual still have to stay restricted to 2024-2026.
daily = (times[times['year'].astype(int) >= 2017]
    .assign(date=lambda d: d['start_time'].dt.normalize())
    .groupby('date')['trip_id']
    .size()
    .reset_index(name='nr_trips')
    )
daily['year'] = daily['date'].dt.year
daily['rolling_365d'] = daily['nr_trips'].rolling(365, min_periods=1).mean()

daily_recent = daily[daily['date'] >= '2024-01-01'].reset_index(drop=True)

daily_type_src = user_type_times[user_type_times['year'].astype(int) >= 2024].copy()
daily_type_src['date'] = daily_type_src['start_time'].dt.normalize()

daily_type_counts = (daily_type_src
    .groupby(['user_type', 'date'])
    .size()
    .reset_index(name='trips')
    )

def daily_type_series(name):
    return (daily_type_counts[daily_type_counts['user_type'] == name]
        .set_index('date')['trips']
        .reindex(daily_recent['date'], fill_value=0))

member_daily = daily_type_series('Member')
casual_daily = daily_type_series('Casual')


In [645]:
# at ~3500 daily points, individual bars render as a washed-out mass rather than
# distinct columns (same issue any dense daily/multi-year bar chart runs into) -
# a plain line reads far more cleanly at this density, so daily granularity uses
# that instead of bars (monthly/weekly stay bars - this is purely a density
# tradeoff, not a style change for its own sake)
daily_main = daily[daily['year'] < 2026]
daily_last = daily[daily['year'] == 2026]

fig_daily_combined = make_subplots(
    rows=2, cols=2,
    specs=[[{'colspan': 2}, None], [{}, {}]],
    row_heights=[0.5, 0.5],
    horizontal_spacing=0.08,
    vertical_spacing=0.18,
    subplot_titles=('All Riders', 'Member (2024-2026)', 'Casual (2024-2026)'),
)

# top row: daily trips, full history (orange), 2026 year-to-date (grey), + 365-day rolling avg
fig_daily_combined.add_trace(go.Scatter(
    x=daily_main['date'], y=daily_main['nr_trips'],
    mode='lines', line=dict(width=1.2, color='#FF6B4A'),
    name='Trips',
    hovertemplate='<b>Date:</b> %{x|%b %d, %Y}<br><b>Trips:</b> %{y:,.0f}<extra></extra>',
), row=1, col=1)

fig_daily_combined.add_trace(go.Scatter(
    x=daily_last['date'], y=daily_last['nr_trips'],
    mode='lines', line=dict(width=1.2, color='#B0B5BA'),
    showlegend=False,
    hovertemplate='<b>Date:</b> %{x|%b %d, %Y}<br><b>Trips:</b> %{y:,.0f}<extra></extra>',
), row=1, col=1)

fig_daily_combined.add_trace(go.Scatter(
    x=daily['date'],
    y=daily['rolling_365d'],
    mode='lines',
    line=dict(color='#1B3A5C', width=2),
    name='365-day rolling avg',
    hovertemplate='<b>Date:</b> %{x|%b %d, %Y}<br><b>365-day avg:</b> %{y:,.0f}<extra></extra>',
), row=1, col=1)

# bottom row: Member (left), Casual (right), 2024-2026 only (see daily_cell1)
fig_daily_combined.add_trace(go.Scatter(
    x=daily_recent['date'], y=member_daily.values,
    mode='lines', line=dict(width=1.2, color='#D9A440'),
    showlegend=False,
    hovertemplate='<b>Date:</b> %{x|%b %d, %Y}<br><b>Trips:</b> %{y:,.0f}<extra></extra>',
), row=2, col=1)

fig_daily_combined.add_trace(go.Scatter(
    x=daily_recent['date'], y=casual_daily.values,
    mode='lines', line=dict(width=1.2, color='#1B3A5C'),
    showlegend=False,
    hovertemplate='<b>Date:</b> %{x|%b %d, %Y}<br><b>Trips:</b> %{y:,.0f}<extra></extra>',
), row=2, col=2)

fig_daily_combined.update_layout(
    title=dict(text='Trips by Day, 2017-2026', x=0.5),
    showlegend=True,
    legend=dict(orientation='h', yanchor='top', y=-0.1, xanchor='center', x=0.5),
    hoverlabel=dict(bgcolor='#FFF8E7', bordercolor='#D9A440', font=dict(color='#1a1a1a', size=13)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    autosize=True,
    margin=dict(t=100, b=80),
)

# top row spans 9 years - yearly ticks; bottom rows span 3 years - half-yearly ticks
fig_daily_combined.update_xaxes(title='Date', showgrid=False, showline=True, linecolor='black', dtick='M12', tickformat='%Y', row=1, col=1)
fig_daily_combined.update_xaxes(title='Date', showgrid=False, showline=True, linecolor='black', dtick='M6', tickformat='%b %Y', row=2, col=1)
fig_daily_combined.update_xaxes(title='Date', showgrid=False, showline=True, linecolor='black', dtick='M6', tickformat='%b %Y', row=2, col=2)

type_y_max_daily = max(member_daily.max(), casual_daily.max()) * 1.1

fig_daily_combined.update_yaxes(title='', showgrid=False, tickformat='.2s', showline=True, linecolor='black', row=1, col=1)
fig_daily_combined.update_yaxes(title='', showgrid=False, tickformat='.2s', showline=True, linecolor='black', range=[0, type_y_max_daily], row=2, col=1)
fig_daily_combined.update_yaxes(title='', showgrid=False, tickformat='.2s', showline=True, linecolor='black', range=[0, type_y_max_daily], row=2, col=2)

fig_daily_combined


In [647]:
description_html = '''
'''

chart_div = fig_daily_combined.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_width='100%',
    default_height='100%',
    config={'responsive': True},
)

panel_html = f'''<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  html, body {{ margin: 0; height: 100%; background: #ffffff; }}
  .panel {{
    display: flex;
    flex-direction: column;
    gap: 16px;
    background: #ffffff;
    padding: 24px;
    box-sizing: border-box;
    height: 100%;
  }}
  .text {{
    flex: 0 0 auto;
    color: #1a1a1a;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    font-size: 20px;
    line-height: 1.5;
  }}
  .text p {{ margin: 0 0 12px 0; }}
  .highlight {{ color: #FF6B4A; font-weight: 700; }}
  .chart {{ flex: 1 1 auto; min-height: 0; }}
</style>
</head>
<body>
  <div class="panel">
    <div class="text">{description_html}</div>
    <div class="chart">{chart_div}</div>
  </div>
</body>
</html>
'''

with open('site/site_assets/daily_type_panel.html', 'w') as f:
    f.write(panel_html)

print("saved site/site_assets/daily_type_panel.html")


saved site/site_assets/daily_type_panel.html


### Ward 2026 data prep (feeds the combined ward+stations map below)

In [348]:
# official Toronto 2018 ward numbering
WARD_NUMBERS = {
    'Etobicoke North': 1, 'Etobicoke Centre': 2, 'Etobicoke-Lakeshore': 3,
    'Parkdale-High Park': 4, 'York South-Weston': 5, 'York Centre': 6,
    'Humber River-Black Creek': 7, 'Eglinton-Lawrence': 8, 'Davenport': 9,
    'Spadina-Fort York': 10, 'University-Rosedale': 11, "Toronto-St. Paul's": 12,
    'Toronto Centre': 13, 'Toronto-Danforth': 14, 'Don Valley West': 15,
    'Don Valley East': 16, 'Don Valley North': 17, 'Willowdale': 18,
    'Beaches-East York': 19, 'Scarborough Southwest': 20, 'Scarborough Centre': 21,
    'Scarborough-Agincourt': 22, 'Scarborough North': 23, 'Scarborough-Guildwood': 24,
    'Scarborough-Rouge Park': 25,
}

df_2026 = df[df['year'] == 2026]

# first year a station existed in each ward, for the tooltip heading
ward_first_year = (
    stations.set_index('station_id')
    .join(first_year, how='inner')
    .groupby('ward')['first_year'].min()
    .reindex(wards['ward'])
)

trips_by_ward_2026 = (
    df_2026.groupby('start_ward').size()
    .reindex(wards['ward']).fillna(0).astype(int)
)
stations_by_ward_2026 = (
    df_2026.groupby('start_ward')['start_station_id'].nunique()
    .reindex(wards['ward']).fillna(0).astype(int)
)
# raw trip counts are dominated by downtown (~6,450x spread between busiest and
# quietest ward) purely because downtown has far more stations, not because each
# station there is used that much harder -> normalize by station count per ward.
# that narrows the spread to ~170x; still wide, so color (not the displayed count)
# is additionally log-scaled for legibility, same approach as the ward timelapse map.
trips_per_station_2026 = (trips_by_ward_2026 / stations_by_ward_2026.replace(0, pd.NA)).round(1)

# simplify boundary detail for a cleaner, blockier look (matches the reference map's style)
wards_simple = wards.copy()
wards_simple['geometry'] = wards_simple.geometry.simplify(tolerance=0.0012, preserve_topology=True)
wards_simple['centroid'] = wards_simple.geometry.centroid

log_tps = np.log(trips_per_station_2026.astype(float))
colormap_wards_2026 = cm.LinearColormap(
    colors=['#EDD9F0', '#9B59B6', '#4A235A'],
    vmin=log_tps.min(),
    vmax=log_tps.max(),
)
colormap_wards_2026.caption = 'Trips per station, 2026 (log scale)'

print(trips_by_ward_2026.to_frame('trips').join(stations_by_ward_2026.rename('stations')).join(trips_per_station_2026.rename('trips_per_station')).sort_values('trips', ascending=False))

                            trips  stations  trips_per_station
ward                                                          
Spadina-Fort York         2727692       187            14586.6
University-Rosedale       1337004       120            11141.7
Toronto Centre            1314137       103            12758.6
Toronto-Danforth           394369        76             5189.1
Davenport                  313052        64             4891.4
Parkdale-High Park         287015        68             4220.8
Toronto-St. Paul's         129342        60             2155.7
Beaches-East York          106150        52             2041.3
Etobicoke-Lakeshore         87066        38             2291.2
Don Valley West             30074        32              939.8
Eglinton-Lawrence           19231        27              712.3
York South-Weston           15876        34              466.9
Scarborough Southwest       15342        39              393.4
Willowdale                  10895        14            

/var/folders/xf/ygl5_zk57kn1rh2h1xj9fyx40000gn/T/ipykernel_1611/4242813924.py:42: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  wards_simple['centroid'] = wards_simple.geometry.centroid


In [ ]:
wards_simple['number'] = wards_simple['ward'].map(WARD_NUMBERS)
wards_simple['trips_2026'] = wards_simple['ward'].map(trips_by_ward_2026)
wards_simple['stations_2026'] = wards_simple['ward'].map(stations_by_ward_2026)
wards_simple['trips_per_station_2026'] = wards_simple['ward'].map(trips_per_station_2026)
wards_simple['color_value'] = wards_simple['ward'].map(log_tps)
wards_simple['first_year'] = wards_simple['ward'].map(ward_first_year)


def _tooltip_html(row):
    return (
        f'<div style="font-weight:700; font-size:15px; margin-bottom:6px;">'
        f'{row["ward"]} ({int(row["first_year"])})</div>'
        '<table style="border-collapse:collapse;">'
        f'<tr><td style="font-weight:700; padding-right:16px;">Rides</td>'
        f'<td>{row["trips_2026"]:,}</td></tr>'
        f'<tr><td style="font-weight:700; padding-right:16px;">Stations</td>'
        f'<td>{row["stations_2026"]:,}</td></tr>'
        f'<tr><td style="font-weight:700; padding-right:16px;">Rides per Station</td>'
        f'<td>{row["trips_per_station_2026"]:,.1f}</td></tr>'
        '</table>'
    )

wards_simple['tooltip_html'] = wards_simple.apply(_tooltip_html, axis=1)

wards_geojson = wards_simple[
    ['ward', 'trips_2026', 'stations_2026', 'trips_per_station_2026', 'color_value',
     'tooltip_html', 'geometry']
].__geo_interface__

### combined map with dots and wards

In [ ]:
station_trips_2026 = df_2026.groupby('start_station_id').size().rename('trips_2026')

stations_2026_pts = (
    stations_full.set_index('station_id')
    .join(station_trips_2026, how='inner')
    .dropna(subset=['lat', 'lon'])
    .reset_index()
)

scaled = np.sqrt(stations_2026_pts['trips_2026'])
scaled_norm = (scaled - scaled.min()) / (scaled.max() - scaled.min())

In [ ]:

stations_2026_pts['radius_px'] = MIN_RADIUS_PX + scaled_norm * (MAX_RADIUS_PX - MIN_RADIUS_PX)

m_combined = folium.Map(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ',
    zoom_control=True,
    scrollWheelZoom=False,
    doubleClickZoom=False,
    touchZoom=False,
    boxZoom=False,
    keyboard=False,
)

# ward layer first (bottom), lighter fill so the station dots stay visible on top of it
folium.GeoJson(
    wards_geojson,
    style_function=lambda feature: {
        'fillOpacity': 0,
        'color': '#1B3A5C',
        'weight': 1.5,
    },
    highlight_function=lambda feature: {'weight': 3, 'color': '#1B3A5C', 'fillColor': '#1B3A5C', 'fillOpacity': 0.15},
    tooltip=folium.GeoJsonTooltip(
        fields=['tooltip_html'],
        labels=False,
        sticky=True,
        style=(
            'background-color: white; border: none; border-radius: 10px; '
            'box-shadow: 0 2px 10px rgba(0,0,0,0.25); padding: 12px 16px; '
            "font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; "
            'font-size: 14px; color: #1a1a1a;'
        ),
    ),
).add_to(m_combined)

# station dots on top
for row in stations_2026_pts.itertuples():
    folium.CircleMarker(
        location=[row.lat, row.lon],
        radius=row.radius_px,
        fill=True,
        fill_color=DOT_COLOR,
        fill_opacity=0.85,
        stroke=True,
        color='white',
        weight=1,
        tooltip=f"{row.name}: {row.trips_2026:,} rides (2026)",
    ).add_to(m_combined)

bounds_wards = [float(v) for v in wards_simple.total_bounds]
m_combined.fit_bounds([[bounds_wards[1], bounds_wards[0]], [bounds_wards[3], bounds_wards[2]]])

m_combined.save('site/site_assets/ward_and_stations_map_2026.html')
print("saved site/site_assets/ward_and_stations_map_2026.html")

## Busiest stations and the most popular routes.

In [ ]:
# top-3 busiest stations in the ward

In [465]:
trips_routes = pd.read_parquet(DATA_DIR / 'trips', columns=['start_station_id', 'end_station_id'])

In [466]:
busiest_stations = (
    trips_routes['start_station_id'].value_counts().head(5)
    .rename_axis('station_id').reset_index(name='trips')
)
busiest_stations['name'] = busiest_stations['station_id'].map(stations_full.set_index('station_id')['name'])
print(busiest_stations)


   station_id   trips                                     name
0        7076  377997                  York St / Queens Quay W
1        7006  301047          Bay St / College St (East Side)
2        7016  288806  Bay St / Queens Quay W (Ferry Terminal)
3        7030  265546                  Bay St / Wellesley St W
4        7033  263041                            Union Station


In [468]:
route_pairs = (
    trips_routes[trips_routes['start_station_id'] != trips_routes['end_station_id']]
    .groupby(['start_station_id', 'end_station_id']).size()
    .sort_values(ascending=False).head(5)
)

In [46]:
# stations_full.set_index('station_id')['name']

In [469]:
name_lookup = stations_full.set_index('station_id')['name']
for (s, e), cnt in route_pairs.items():
    print(cnt, name_lookup.get(s, s), '->', name_lookup.get(e, e))

34994 Centre Island Ferry Dock -> Ward's Island Ferry Dock
19544 Ward's Island Ferry Dock -> Centre Island Ferry Dock
19067 Centre Island Ferry Dock -> Centre Island
18225 Centre Island -> Ward's Island Ferry Dock
17513 Centre Island -> Centre Island Ferry Dock


In [634]:
busiest_rows = ''.join(
    f'<tr><td>{i+1}</td><td>{row["name"]}</td><td>{row["trips"]:,}</td></tr>'
    for i, row in busiest_stations.reset_index(drop=True).iterrows()
)

route_rows = ''
for (s, e), cnt in route_pairs.items():
    route_rows += f'<tr><td>{name_lookup.get(s, s)} → {name_lookup.get(e, e)}</td><td>{cnt:,}</td></tr>'

description_html = '''
<p><span class="highlight">York St / Queens Quay W</span> is the single busiest station in the system — over 376,000 trips to date, anchored by the financial district and the waterfront.</p>
<p>But look at the busiest <em>routes</em> instead of busiest stations, and the story flips: four of the five most popular point-to-point trips shuttle back and forth between the Centre Island and Ward's Island ferry docks — a loop almost nobody would ride to get to work. It's day-trippers circling the Toronto Islands, not commuters.</p>
'''

panel_html = f'''<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  html, body {{ margin: 0; height: 100%; background: #ffffff; }}
  .panel {{
    display: flex;
    gap: 32px;
    background: #ffffff;
    padding: 24px;
    box-sizing: border-box;
    height: 100%;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    color: #1a1a1a;
  }}
  .text {{ flex: 2; min-width: 0; font-size: 20px; line-height: 1.5; }}
  .text p {{ margin: 0 0 20px 0; }}
  .highlight {{ color: #FF6B4A; font-weight: 700; }}
  .tables {{ flex: 3; min-width: 0; display: flex; flex-direction: column; gap: 20px; justify-content: center; }}
  table {{ width: 100%; border-collapse: collapse; font-size: 15px; }}
  th {{ text-align: left; border-bottom: 2px solid #1a1a1a; padding: 6px 8px; }}
  td {{ padding: 6px 8px; border-bottom: 1px solid #e0e0e0; }}
  caption {{ text-align: left; font-weight: 700; margin-bottom: 8px; font-size: 16px; }}
</style>
</head>
<body>
  <div class="panel">
    <div class="text">{description_html}</div>
    <div class="tables">
      <table>
        <caption>Busiest stations</caption>
        <tr><th>#</th><th>Station</th><th>Trips</th></tr>
        {busiest_rows}
      </table>
      <table>
        <caption>Busiest routes</caption>
        <tr><th>Route</th><th>Trips</th></tr>
        {route_rows}
      </table>
    </div>
  </div>
</body>
</html>
'''

with open('site/site_assets/busiest_stations_routes_panel.html', 'w') as f:
    f.write(panel_html)

print("saved site/site_assets/busiest_stations_routes_panel.html")

saved site/site_assets/busiest_stations_routes_panel.html


## Users

In [215]:
users = pd.read_parquet(DATA_DIR / 'trips', columns=['trip_id', 'year', 'user_type'])

In [216]:
trips_usertype = (users
    .groupby(['year', 'user_type'])['trip_id']
    .size()
    .reset_index(name = 'nr_trips')
    )

In [217]:
trips_usertype.head() 

,year,user_type,nr_trips
0,2016,Casual,122484
1,2016,Member,245477
2,2017,Casual,327585
3,2017,Member,1164782
4,2018,Annual Member,1572980


# Some ratios 

In [166]:
trips.head()

,trip_id,start_station_id,year
0,53279,7100,2016
1,53394,7097,2016
2,58314,7100,2016
3,60784,7033,2016
4,93164,7059,2016


### trips per day

In [371]:
trips_per_day = (trips
    .groupby('year')
    .size()
    .reset_index(name='trips')
    .assign(trips_per_day = lambda d: round(d['trips']/365))
    )

trips_per_day

,year,trips,trips_per_day
0,2016,585530,1604.0
1,2017,1492366,4089.0
2,2018,1922955,5268.0
3,2019,2439448,6683.0
4,2020,2909384,7971.0
5,2021,3569606,9780.0
6,2022,4615878,12646.0
7,2023,5708745,15640.0
8,2024,6942221,19020.0
9,2025,7807041,21389.0


In [158]:
times.head()

,trip_id,year,start_time,end_time,trip_duration
0,53279,2016,2016-07-09 01:03:00,2016-07-09 01:15:00,714
1,53394,2016,2016-07-09 02:15:00,2016-07-09 02:22:00,417
2,58314,2016,2016-07-10 17:04:00,2016-07-10 17:36:00,1904
3,60784,2016,2016-07-11 01:45:00,2016-07-11 01:58:00,784
4,93164,2016,2016-07-18 13:35:00,2016-07-18 13:42:00,443


In [ ]:
times

In [ ]:
times.start_time.

0            9
1            9
2           10
3           11
4           18
            ..
44638683    30
44638684    30
44638685    30
44638686    30
44638687    30
Name: start_time, Length: 44638688, dtype: int32